# EDA Fakeddit (sous-ensemble)

Avant de lancer : Add Input -> ton dataset `fakeddit-subset`. Lance les cellules dans l'ordre et copie les sorties des cellules 2 a 6.

In [ ]:
# Cellule 1 : imports et detection automatique des chemins
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Colab : remplace par P = '/content/' et IMG = '/content/images/'
hits = glob.glob('/kaggle/input/**/train.tsv', recursive=True)
assert hits, 'train.tsv introuvable : ajoute ton dataset avec Add Input'
P = os.path.dirname(hits[0]) + '/'
IMG = P + 'images/images/' if os.path.isdir(P + 'images/images') else P + 'images/'
print('P   =', P)
print('IMG =', IMG)

In [ ]:
# Cellule 2 : chargement et controle d'integrite (on veut 0 image manquante)
tr, va, te = [pd.read_csv(P + s + '.tsv', sep='\t') for s in ['train', 'val', 'test']]
have = {f[:-4] for f in os.listdir(IMG)}
print(len(have), 'images dans le dossier (attendu : environ 8196)')
for name, d in [('train', tr), ('val', va), ('test', te)]:
    missing = (~d['id'].astype(str).isin(have)).sum()
    print(name, len(d), 'exemples | images manquantes :', missing)

In [ ]:
# Cellule 3 : equilibre des classes et longueur des titres
for name, d in [('train', tr), ('val', va), ('test', te)]:
    print(name, len(d), d['2_way_label'].value_counts(normalize=True).round(3).to_dict())
    print('   6-way :', d['6_way_label'].value_counts().sort_index().to_dict())

tr['n_words'] = tr['clean_title'].str.split().str.len()
print(tr.groupby('2_way_label')['n_words'].describe().round(1))
for lab in [0, 1]:
    plt.hist(tr.loc[tr['2_way_label'] == lab, 'n_words'], bins=30, alpha=0.5, label='classe ' + str(lab))
plt.xlabel('mots par titre'); plt.legend(); plt.show()

In [ ]:
# Cellule 4 : subreddits (utile pour RQ4 et pour reperer les raccourcis)
sub = tr.groupby(['subreddit', '2_way_label']).size().unstack(fill_value=0)
sub['total'] = sub.sum(axis=1)
sub['part_classe1'] = (sub[1] / sub['total']).round(2)
print(sub.sort_values('total', ascending=False).head(15))
print('nombre de subreddits (train) :', tr['subreddit'].nunique())
print('subreddits du test absents du train :', set(te['subreddit']) - set(tr['subreddit']))

In [ ]:
# Cellule 5 : periode temporelle (plan B pour la generalisation)
for name, d in [('train', tr), ('val', va), ('test', te)]:
    t = pd.to_datetime(d['created_utc'], unit='s')
    print(name, t.min().date(), '->', t.max().date())

In [ ]:
# Cellule 6 : fuite de donnees et doublons
print('titres communs train/test :', len(set(tr['clean_title']) & set(te['clean_title'])))
print('titres communs train/val  :', len(set(tr['clean_title']) & set(va['clean_title'])))
print('doublons dans le train    :', tr['clean_title'].duplicated().sum())
print('ids communs train/test    :', len(set(tr['id']) & set(te['id'])))

In [ ]:
# Cellule 7 : grille d'images, 6 exemples par classe
fig, ax = plt.subplots(2, 6, figsize=(16, 6))
for r, lab in enumerate([0, 1]):
    ech = tr[tr['2_way_label'] == lab].sample(6, random_state=1)
    for c, (_, row) in enumerate(ech.iterrows()):
        ax[r][c].imshow(Image.open(IMG + str(row['id']) + '.jpg'))
        ax[r][c].axis('off')
        ax[r][c].set_title(str(lab) + ' | ' + row['clean_title'][:28], fontsize=7)
plt.tight_layout(); plt.show()